# 02 — Understat Data Collection and Integration

This notebook combines the five league-level Understat Excel files into one dataset.

**Input:** five original league workbooks in `data/raw/understat/`  
**Output:** `data/processed/Understat_dataset.xlsx`

The steps preserve the original dissertation workflow: load each league, add a league identifier, concatenate the records, remove the source ranking column (`number`), and drop rows with missing values.

> **Scope note:** The supplied workbooks contain seasons from 2019–20 through 2025–26. This notebook does not filter seasons; downstream notebooks must apply the dissertation observation window (2019–20 to 2023–24) where appropriate.


In [1]:
from pathlib import Path
import pandas as pd

# Resolve the project root whether this notebook is run from the root or notebooks/
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

RAW_UNDERSTAT_DIR = PROJECT_ROOT / "data" / "raw" / "understat"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)


In [2]:
# Map each source workbook to its league name
understat_files = {
    "Bundesliga X(g).xlsx": "Bundesliga",
    "EPL X(g).xlsx": "Premier League",
    "La liga X(g).xlsx": "La Liga",
    "Ligue 1 X(g).xlsx": "Ligue 1",
    "Serie A X(g).xlsx": "Serie A",
}

missing_files = [
    filename for filename in understat_files
    if not (RAW_UNDERSTAT_DIR / filename).exists()
]
if missing_files:
    raise FileNotFoundError(
        "The following Understat source files are missing from "
        f"{RAW_UNDERSTAT_DIR}: {missing_files}"
    )


In [3]:
# Load each league workbook and attach its league name
league_frames = []

for filename, league_name in understat_files.items():
    league_df = pd.read_excel(RAW_UNDERSTAT_DIR / filename)
    league_df["league"] = league_name
    league_frames.append(league_df)

# Combine all five leagues
understat = pd.concat(league_frames, ignore_index=True)

print(f"Combined rows before cleaning: {len(understat):,}")
print(f"Combined columns before cleaning: {len(understat.columns):,}")


Combined rows before cleaning: 18,827
Combined columns before cleaning: 19


In [4]:
# Remove the source ranking column and rows containing missing values
understat = understat.drop(columns=["number"])
rows_before_dropna = len(understat)
understat = understat.dropna().copy()

print(f"Rows removed due to missing values: {rows_before_dropna - len(understat):,}")
print(f"Final rows: {len(understat):,}")
print(f"Final columns: {len(understat.columns):,}")


Rows removed due to missing values: 0
Final rows: 18,827
Final columns: 18


In [5]:
# Check coverage by league and season
display(
    understat.groupby(["league", "Season"])
    .size()
    .unstack(fill_value=0)
)

display(understat.head())


Season,2019 - 2020,2020 - 2021,2021 - 2022,2022 - 2023,2023 - 2024,2024 - 2025,2025 - 2026
league,,,,,,,
Bundesliga,487,496,511,506,494,481,499
La Liga,556,570,603,584,598,590,600
Ligue 1,531,575,589,585,525,542,553
Premier League,515,537,537,554,570,562,537
Serie A,572,594,608,577,590,0,599


,player,team,min,NPG,xG,NPxG,xA,xGChain,xGBuildup,xG90,NPxG90,xA90,xG90xA90,NPxG90xA90,xGChain90,xGBuildup90,Season,league
0,Robert Lewandowski,Bayern Munich,2467,33,32.08,25.26,4.82,31.74,5.69,1.17,0.92,0.18,1.35,1.10,1.16,0.21,2020 - 2021,Bundesliga
1,André Silva,Eintracht Frankfurt,2787,21,25.60,20.29,5.47,26.75,4.10,0.83,0.66,0.18,1.00,0.83,0.86,0.13,2020 - 2021,Bundesliga
2,Erling Haaland,Borussia Dortmund,2416,25,23.60,20.57,4.04,27.20,5.90,0.88,0.77,0.15,1.03,0.92,1.01,0.22,2020 - 2021,Bundesliga
3,Andrej Kramaric,Hoffenheim,2386,15,15.53,11.74,4.07,18.06,5.37,0.59,0.44,0.15,0.74,0.60,0.68,0.20,2020 - 2021,Bundesliga
4,Wout Weghorst,Wolfsburg,2954,18,18.31,16.04,5.43,24.29,5.95,0.56,0.49,0.17,0.72,0.65,0.74,0.18,2020 - 2021,Bundesliga


In [6]:
# Save the combined dataset
output_path = PROCESSED_DIR / "Understat_dataset.xlsx"
understat.to_excel(output_path, index=False)

print(f"Saved combined Understat dataset to: {output_path}")


Saved combined Understat dataset to: /mnt/data/football-player-development-prediction-repro/data/processed/Understat_dataset.xlsx


## Notes for reproducibility

- The five original Excel files must be placed in `data/raw/understat/`.
- The original notebook removes every row with at least one missing value. This behaviour has been retained to preserve the original pipeline.
- The supplied source files cover seven seasons (2019–20 to 2025–26). The dissertation's modelling scope is narrower, so season filtering should be performed explicitly in the relevant downstream preparation step.
- Check data-provider terms before redistributing the raw files or the combined dataset.
